In [1]:
!pip install -q spacy altair GPUtil 

In [2]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 88.0 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 118.5 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import os
from os.path import exists

import sys
import math
import copy
import time
import spacy
import GPUtil

In [4]:
from pathlib import Path
import pandas as pd
import altair as alt

In [ ]:
import urllib.request
import shutil
import tarfile
from collections import Counter
from itertools import chain

In [6]:
import torch

import torch.nn as nn
from torch.nn.functional import log_softmax, pad
from torch.nn.parallel import DistributedDataParallel as DDP # Cf., Lecture 12

from torch.optim.lr_scheduler import LambdaLR

from torch.utils.data import DataLoader
from torch.utils.data.distributed import DistributedSampler

import torch.distributed as dist
import torch.multiprocessing as mp

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
RUN_EXAMPLES = True

def is_interactive_notebook():
    return __name__ == "__name__"

def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)

def execute_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        fn(*args)

In [9]:
class DummyOptimizer(torch.optim.Optimizer):
    def __init__(self):
        self.param_groups = [{"lr":0}]
        None
    def step(self):
        None
    def zero_grad(self, set_to_none = False):
        None

class DummyScheduler:
    def step(self):
        None

# Part 1: Model Architecture

## Encoder-Decoder

In [10]:
class EncoderDecoder(nn.Module):
    """
    A Standard Encoder-Decoder Architecture.
    """

    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super(EncoderDecoder, self).__init__()

        self.encoder = encoder
        self.decoder = decoder

        self.src_embed = src_embed
        self.tgt_embed = tgt_embed

        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        return self.decode(self.encode(src, src_mask), src_mask, tgt, tgt_mask)
    
    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)

In [ ]:
class Generator(nn.Module):

    def __init__(self, d_model, vocab):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab)
    
    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

## Encoder

In [ ]:
def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])

In [ ]:
class Encoder(nn.Module):

    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size) 

    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

In [ ]:
class LayerNorm(nn.Module):

    def __init__(self, features, eps = 1e-6):
        super(LayerNorm, self).__init__()
        self.a_2 = nn.Parameter(torch.ones(features)) # Scale, 학습 가능 파라미터, size는 feature와 동일 (즉, Transformer에서는 512)
        self.b_2 = nn.Parameter(torch.zeros(features)) # Bias, 학습 가능 파라미터
        self.eps = eps # epsilon, 작은 양수값, std -> 0 일 때, 나누는 값이 0이 되지 않도록 더해주는 용도

    def forward(self, x):
        mean = x.mean(-1, keepdim=True) 
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2

In [ ]:
class SublayerConnection(nn.Module):
    
    def __init__(self, size, dropout):
        super(SublayerConnection, self).__init__()
        self.norm = LayerNorm(size) 
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))

In [ ]:
class EncoderLayer(nn.Module):

    def __init__(self, size, self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = self_attn 
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size

    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask)) # self_attn(query, key, value, mask)
        return self.sublayer[1](x, self.feed_forward)

## Decoder

In [ ]:
class Decoder(nn.Module):

    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)

    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm

In [ ]:
class DecoderLayer(nn.Module):

    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)

    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory # 인코더에는 memory가 포함되지 않았음
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        # self_attn(query, key, value, mask),
        # 따라 두번째 sub-layer는 q는 이전 decoder의 sub-layer에서 출력된 값을 가지지만, k와 v는 인코더로부터 값을 가져오게 됨
        x = self.sublayer[1](x, lambda x: self.self_attn(x, m, m, tgt_mask)) 
        return self.sublayer[2](x, self.feed_forward)

In [ ]:
def subsequent_mask(size):
    attn_shape = (1, size, size)
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal = 1).type(
        torch.unit8
    )
    return subsequent_mask == 0

## Attention

In [ ]:
def attention(query, key, value, mask = None, dropout = None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) /math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)

    p_attn = scores.softmax(dim=-1) 

    if dropout is not None:
        p_attn = dropout(p_attn) # dropout이 function으로 입력되는가?

    return torch.matmul(p_attn, value), p_attn


In [ ]:
class MultiHeadedAttention(nn.Module):

    def __init__(self, h, d_model, dropout = 0.1):
        # model size와 head의 개수(h)를 입력받음
        super(MultiHeadedAttention, self).__init__()
        assert d_model % h == 0

        self.d_k = d_model // h
        self.h = h
        self.linears = clones(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
